# PCam binary classifier — ResNet-18 on Kaggle GPU

**Task:** Binary classification of 96×96 histopathology patches (PatchCamelyon).  
Tumour-positive patches contain metastatic tissue in the central 32×32 region.

**Benchmark context:**  
PCam is a well-studied benchmark. Top published results reach 0.97–0.99 AUC.  
ResNet-18 with a proper training recipe routinely achieves 0.96+ — no exotic architecture needed.  
A naive baseline (flat LR, basic augmentation, 5 epochs) typically lands around 0.94.

**Every choice in this notebook is deliberate. For each one we document:**  
- What the old/common approach is
- Why this choice is better
- Where it comes from in the literature

| Choice | Old way | This notebook |
|--------|---------|---------------|
| Optimiser | `Adam` | `AdamW` — correct weight decay (Loshchilov & Hutter, 2019) |
| LR schedule | Constant 1e-4 | Cosine annealing — smooth decay, no manual step tuning |
| Loss | `CrossEntropyLoss` | + `label_smoothing=0.1` — prevents overconfident predictions |
| Augmentation | Flip only | + stain jitter, blur — histopathology-specific |
| Rotation aug | `RandomRotation(180)` — bilinear interp per sample | `RandomRot90` — `torch.rot90` zero-copy view, same D4 coverage |
| Precision | float32 everywhere | Mixed precision (AMP) — ~2× on T4 (Tensor Cores), ~1.3× on P100 |
| Stopping | Fixed epoch count | Early stopping on AUC — stops when generalisation plateaus |
| Threshold | Default 0.5 | Tuned via Youden's J — addresses the false-negative problem |
| CPU pipeline | v1 transforms (PIL per sample) | v2 tensor-native — no PIL, `persistent_workers`, `prefetch_factor=4` |
| Memory format | NCHW (default) | Channels-last (NHWC) — 10–20% faster conv on T4 Tensor Cores |
| Multi-GPU (T4x2) | Single GPU only | `DataParallel` + linear batch/LR scaling — ~2× throughput |
| Single GPU | Eager execution | `torch.compile` — kernel fusion, 10–20% speedup after warm-up |

**Dataset layout (andrewmvd/metastatic-tissue-classification-patchcamelyon)**
```
pcam/training_split.h5         key='x'  shape (262144, 96, 96, 3) uint8
pcam/validation_split.h5       key='x'  shape  (32768, 96, 96, 3) uint8
Labels/Labels/..._train_y.h5   key='y'  shape (262144, 1, 1, 1)   int
Labels/Labels/..._valid_y.h5   key='y'  shape  (32768, 1, 1, 1)   int
```

In [ ]:
# ── 0. Fix PyTorch version ────────────────────────────────────────────────────
# Kaggle ships PyTorch 2.10.0+cu128 which dropped support for older GPU
# architectures (P100=sm_60, T4=sm_75). cu118 covers sm_60–sm_90 — any GPU
# Kaggle may assign. Kernel restarts automatically after install.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.4.1+cu118', 'torchvision==0.19.1+cu118',
    '--index-url', 'https://download.pytorch.org/whl/cu118',
], check=True)
print('Install complete — restarting kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ── 1. Environment ────────────────────────────────────────────────────────────
import os, time, json, logging
from dataclasses import dataclass, asdict
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import v2
# ↑ torchvision.transforms.v2 (introduced in 0.15, stable in 0.19) is the
# modern replacement for torchvision.transforms (v1).
# Old way (v1): transforms expected PIL Images. Every sample required:
#   numpy array → PIL Image (Image.fromarray) → transform ops → tensor
# New way (v2): transforms operate natively on torch.uint8 tensors.
#   numpy array → torch.uint8 tensor (once, at load time) → transform ops
# The PIL step is eliminated entirely from the hot path (~262k calls/epoch).
# v2 also supports ToDtype() which replaces ToTensor() + manual scaling.
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

# Old way: nothing — cuDNN picks a default convolution algorithm at runtime.
# With benchmark=True: on the very first batch, cuDNN runs a short profiling
# pass trying several convolution implementations (FFT, Winograd, implicit GEMM
# etc.) and permanently selects the fastest one for this input shape (96×96×3).
# Subsequent batches use the optimal algorithm with no overhead.
# Only valid when input size is constant — which it is here (all patches 96×96).
torch.backends.cudnn.benchmark = True

# ── GPU capability detection ───────────────────────────────────────────────────
# Kaggle can assign P100 (sm_60, Pascal) or T4 (sm_75, Turing) or T4x2.
# The capability major version determines which GPU features are available.
N_GPUS = torch.cuda.device_count()

if N_GPUS > 0:
    cap = torch.cuda.get_device_capability(0)
    # Tensor Cores exist on Volta (sm_70) and newer — T4 (sm_75) qualifies.
    # On Tensor Core GPUs: float16 matmul and convolutions use dedicated
    # hardware units, giving a larger AMP speedup than pure bandwidth savings.
    # P100 (sm_60) has no Tensor Cores — AMP still helps by halving memory
    # bandwidth (fp16 reads 2× faster), just not as dramatically as on T4.
    HAS_TENSOR_CORES = cap[0] >= 7
    gpu_name = torch.cuda.get_device_name(0)
    print(f'PyTorch       : {torch.__version__}')
    print(f'GPU count     : {N_GPUS}× {gpu_name}')
    print(f'Compute cap   : sm_{cap[0]}{cap[1]}')
    print(f'Tensor Cores  : {"YES — T4 path (channels-last + compile)" if HAS_TENSOR_CORES else "NO  — P100 path (AMP bandwidth savings only)"}')
    print(f'Multi-GPU     : {"YES — DataParallel will scale batch + LR" if N_GPUS > 1 else "NO  — single GPU"}')
    print(f'VRAM (GPU 0)  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    HAS_TENSOR_CORES = False
    print(f'PyTorch : {torch.__version__}  |  CUDA: NOT available')

# Smoke test — catches CUDA compatibility issues before loading 7 GB of data
t = torch.ones(2, 3, 96, 96).cuda()
print(f'\nGPU smoke test: {t.shape} ✓')
del t

In [ ]:
# ── 2. Paths ──────────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon')

TRAIN_X = BASE / 'pcam/training_split.h5'
TRAIN_Y = BASE / 'Labels/Labels/camelyonpatch_level_2_split_train_y.h5'
VAL_X   = BASE / 'pcam/validation_split.h5'
VAL_Y   = BASE / 'Labels/Labels/camelyonpatch_level_2_split_valid_y.h5'

OUTPUT_DIR = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_X, TRAIN_Y, VAL_X, VAL_Y]:
    assert p.exists(), f'Missing: {p}'
    print(f'OK  {p}')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class TrainingConfig:
    output_dir: str = '/kaggle/working/checkpoints'

    # Old way: fixed small epoch count (e.g. 5).
    # This approach: set a generous ceiling and rely on early stopping.
    # Early stopping monitors val AUC and halts when it stops improving,
    # avoiding both underfitting (too few epochs) and overfitting (too many).
    epochs:   int   = 20
    patience: int   = 4

    # batch_size here means per-GPU batch size.
    # When using DataParallel on T4x2, the DataLoader will receive
    # batch_size × N_GPUS so that each GPU still sees 128 samples per step.
    # ResNet-18 fits 128 samples comfortably in 16 GB at float16.
    batch_size: int = 128

    # Slightly higher LR than the naive 1e-4 baseline — works better with
    # cosine annealing because the schedule will decay it smoothly anyway.
    # For multi-GPU, this is scaled up in cell-run (linear scaling rule).
    learning_rate: float = 3e-4

    # Old way: Adam with no weight_decay, or Adam with weight_decay (wrong).
    # Adam applies weight decay as L2 regularisation on the gradient update,
    # which interacts with the adaptive learning rate in an unintended way.
    # AdamW decouples weight decay from the gradient, applying it directly
    # to the weights. This is the correct formulation and generalises better.
    # Reference: Loshchilov & Hutter, "Decoupled Weight Decay Regularization",
    # ICLR 2019 — now the default in transformers, ViT, and most modern work.
    weight_decay: float = 1e-4

    # Old way: CrossEntropyLoss with hard targets (0 or 1).
    # Label smoothing replaces hard targets with soft ones:
    #   1.0 → 1 - smoothing + smoothing/n_classes = 0.9 + 0.05 = 0.95
    #   0.0 → smoothing/n_classes = 0.05
    # This prevents the model from becoming overconfident, which matters here
    # because we use the output probabilities for threshold tuning.
    # Introduced in Inception-v3 (Szegedy et al., 2016); now standard.
    label_smoothing: float = 0.1

    # Old way: no gradient clipping.
    # Gradient clipping caps the global gradient norm to max_norm before
    # the update step. Prevents a single noisy batch from destabilising
    # training with a very large parameter update.
    # max_norm=1.0 is the standard value used in BERT, GPT, ViT, etc.
    grad_clip: float = 1.0

    # Old way: hardcoded num_workers=2.
    # Kaggle allocates 4 CPU cores per notebook. Using all available cores
    # keeps the GPU fed without manually tuning this per environment.
    # min() cap prevents over-subscription if somehow cpu_count() is large.
    num_workers:       int = min(4, os.cpu_count() or 2)
    log_every_n_steps: int = 200

    @property
    def device(self) -> torch.device:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = TrainingConfig()
print(cfg)

In [ ]:
# ── 4. Dataset + augmentation ─────────────────────────────────────────────────

class PCamDataset(Dataset):
    """
    CPU pipeline — old way vs this approach:

    OLD WAY (v1 transforms + PIL):
      __init__:    load numpy array into RAM
      __getitem__: numpy[idx] → Image.fromarray() → v1 transforms → tensor
                   ↑ PIL conversion happens 262,144 times per epoch
                   ↑ PIL ops (rotate, jitter) are single-threaded C code
                   ↑ ToTensor() does an extra copy + type conversion

    THIS WAY (v2 transforms + torch.uint8):
      __init__:    load numpy array → convert to torch.uint8 tensor once
      __getitem__: tensor[idx].permute() → v2 transforms → float32 tensor
                   ↑ no PIL anywhere — pure tensor ops, parallelisable
                   ↑ ToDtype(float32, scale=True) replaces ToTensor() in-place
                   ↑ workers share the uint8 tensor read-only via fork (no copy)

    Memory: uint8 is 1 byte/element — same as numpy. float32 would be 4×.
    Loading as uint8 keeps the 7.2 GB footprint manageable.
    """

    def __init__(self, x_path: Path, y_path: Path, transform):
        log.info(f'Loading {x_path.name} into memory...')
        with h5py.File(x_path, 'r') as f:
            # torch.from_numpy shares memory with the numpy array (zero-copy).
            # uint8 dtype preserved — v2 transforms handle the conversion.
            self.x = torch.from_numpy(f['x'][:])          # uint8 (N, 96, 96, 3)
        with h5py.File(y_path, 'r') as f:
            self.y = torch.from_numpy(
                f['y'][:].reshape(-1).astype(np.int64)    # int64 (N,)
            )
        self.transform = transform
        log.info(f'  {len(self.x)} samples — '
                 f'{(self.y==0).sum().item()} normal / {(self.y==1).sum().item()} tumour')

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # (H, W, C) → (C, H, W): torchvision expects channels-first.
        # permute() is a zero-copy view — no data is moved.
        # .contiguous() ensures memory layout is compatible with v2 transforms.
        image = self.x[idx].permute(2, 0, 1).contiguous()  # uint8 (3, 96, 96)
        return self.transform(image), self.y[idx].item()


class RandomRot90:
    """
    Randomly rotate by 0°, 90°, 180°, or 270°.

    OLD WAY: v2.RandomRotation(degrees=180) — arbitrary angle rotation.
      Requires bilinear interpolation (grid_sample): every output pixel is
      computed as a weighted average of 4 input pixels. 262k calls/epoch ×
      96×96×3 pixels each = heavy CPU load. This was pegging all 4 CPU cores
      on Kaggle T4x2, leaving GPU utilisation at only 34–38%.

    THIS WAY: torch.rot90 — discrete 90° steps only.
      torch.rot90 returns a VIEW of the tensor with strides adjusted.
      Zero pixels are recomputed. Zero memory allocated. Negligible CPU cost.

    Coverage: 4 rotations × 2 flips (RandomHorizontalFlip + RandomVerticalFlip)
    = 8 unique orientations = the full dihedral group D4 (all symmetries of
    the square). Same invariance coverage as continuous rotation, zero cost.

    For histopathology: tissue patches have no canonical orientation —
    D4 invariance is sufficient (Tellez et al., Medical Image Analysis, 2019).
    Continuous rotation adds sub-pixel interpolation artefacts with no benefit.
    """
    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        k = torch.randint(0, 4, (1,)).item()   # 0→0°, 1→90°, 2→180°, 3→270°
        return torch.rot90(img, k, dims=[-2, -1])


def get_transforms():
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    train_tf = v2.Compose([
        # Zero-copy discrete rotation — replaces expensive bilinear rotation.
        # See RandomRot90 docstring above for the full rationale.
        RandomRot90(),

        # Together with RandomRot90, flips cover all 8 symmetries of D4.
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip(),

        # H&E staining varies between labs and scanners. ColorJitter makes
        # the model robust to stain shift that it will see in deployment.
        # hue=0.04 is small by design — simulate stain drift, not arbitrary
        # colour changes that would alter tissue appearance semantics.
        v2.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1, hue=0.04),

        # Simulates slight focus variation across the slide surface.
        # p=0.2 — occasional regularisation, not dominant.
        v2.RandomApply([v2.GaussianBlur(kernel_size=3)], p=0.2),

        # OLD WAY (v1): ToTensor() did uint8→float32 + divide by 255 as a
        # separate operation, then Normalize() did a second pass.
        # v2: ToDtype(float32, scale=True) does both in a single fused op —
        # divides by 255 during the dtype conversion, no extra pass needed.
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std),
    ])

    val_tf = v2.Compose([
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=mean, std=std),
    ])

    return train_tf, val_tf


def get_loaders(cfg: TrainingConfig):
    train_tf, val_tf = get_transforms()
    train_ds = PCamDataset(TRAIN_X, TRAIN_Y, train_tf)
    val_ds   = PCamDataset(VAL_X,   VAL_Y,   val_tf)

    # OLD WAY: DataLoader with default persistent_workers=False.
    # Problem: worker processes are killed and respawned at the start of
    # every epoch. On a 20-epoch run this means 20 process spawns, each
    # requiring reimport of modules and re-fork of the dataset tensor.
    # With persistent_workers=True: workers stay alive between epochs,
    # sleeping until the next epoch starts. The tensor stays in worker memory.
    #
    # prefetch_factor=4: each worker keeps 4 batches ready in a queue.
    # Old default was 2. With fast tensor ops the GPU can drain the queue
    # quickly — a larger prefetch buffer prevents the GPU from ever waiting.
    loader_kwargs = dict(
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        pin_memory=True,         # copies batches to page-locked memory → faster H2D transfer
        persistent_workers=True, # workers survive epoch boundaries
        prefetch_factor=4,       # batches queued per worker ahead of consumption
    )
    train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
    val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
    return train_loader, val_loader

In [ ]:
# ── 5. Model ──────────────────────────────────────────────────────────────────
#
# Why ResNet-18 and not something bigger?
# PCam patches are 96×96 — small enough that deeper models offer diminishing
# returns. The original PCam paper (Veeling et al., NeurIPS 2018) showed
# ResNet-18 achieves near-ceiling performance. EfficientNet-B0 or ViT-Small
# can push AUC ~0.1% higher at 3–5× the compute cost. Not worth it here.
#
# Why ImageNet pretrained weights despite the domain gap?
# Natural image pretraining consistently outperforms random init on medical
# images, even for histopathology (Raghu et al., NeurIPS 2019).
# Low-level detectors (edges, textures, colour blobs) transfer well.
# The high-level semantics are corrected by fine-tuning.

def build_model() -> nn.Module:
    model    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)

    # OLD WAY: default NCHW (channels-first) memory layout.
    # CHANNELS-LAST (NHWC): PyTorch stores (N, H, W, C) in memory instead of
    # (N, C, H, W). For convolutions on Tensor Core GPUs (T4, sm_75+), cuDNN
    # has dedicated NHWC kernels that can be 10–20% faster than NCHW because
    # the convolution access pattern aligns better with NHWC memory layout.
    # On P100 (no Tensor Cores) the difference is small, but .to(channels_last)
    # is a no-op performance-wise — never harmful.
    # Input tensors must also be converted (done in train_epoch / evaluate).
    model = model.to(memory_format=torch.channels_last)

    return model

In [ ]:
# ── 6. Train + eval ───────────────────────────────────────────────────────────

def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg) -> float:
    model.train()
    total_loss = 0.0
    device = cfg.device

    for step, (images, labels) in enumerate(loader):
        # Convert inputs to channels-last to match the model's memory format.
        # This avoids an implicit reformat inside the conv layers and ensures
        # cuDNN can use its NHWC kernels end-to-end without intermediate copies.
        images = images.to(device, memory_format=torch.channels_last)
        labels = labels.to(device)
        optimizer.zero_grad()

        # OLD WAY: forward pass in float32 everywhere.
        # Mixed precision (AMP): forward pass runs in float16, which halves
        # memory bandwidth and doubles throughput on Tensor Core GPUs (T4).
        # On P100 (no Tensor Cores) AMP still reduces memory pressure and
        # gives a ~30–40% speedup from bandwidth savings alone.
        # GradScaler prevents float16 underflow by scaling the loss up before
        # backward and scaling gradients back down before the update.
        # torch.autocast with explicit device_type is the modern API (2.0+).
        # The old form `with autocast():` is deprecated and infers device_type
        # from context — explicit is clearer and avoids future warnings.
        with torch.autocast(device_type='cuda'):
            loss = criterion(model(images), labels)

        scaler.scale(loss).backward()

        # Gradient clipping must happen AFTER scaler.unscale_() to work on the
        # true gradient values, not the scaled ones.
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if step % cfg.log_every_n_steps == 0:
            log.info(f'  step {step}/{len(loader)}  loss={loss.item():.4f}  '
                     f'lr={scheduler.get_last_lr()[0]:.2e}')

    # Cosine annealing steps once per epoch, not per batch.
    scheduler.step()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device) -> dict:
    model.eval()
    total_loss, all_labels, all_probs, all_preds = 0.0, [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, memory_format=torch.channels_last)
            labels = labels.to(device)
            with torch.autocast(device_type='cuda'):
                logits = model(images)
            total_loss += criterion(logits, labels).item()
            all_probs.extend(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)
    return {
        'loss':             round(total_loss / len(loader), 4),
        'accuracy':         round(float((all_preds == all_labels).mean()), 4),
        'auc':              round(float(roc_auc_score(all_labels, all_probs)), 4),
        'f1':               round(float(f1_score(all_labels, all_preds)), 4),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist(),
        '_probs':           all_probs,   # internal — used for threshold search
        '_labels':          all_labels,
    }

In [ ]:
# ── 7. Run ────────────────────────────────────────────────────────────────────
device = cfg.device

# ── GPU-adaptive batch size and learning rate ──────────────────────────────
# With DataParallel on T4x2, the DataLoader feeds the full total_batch to
# DataParallel, which splits it evenly across GPUs. Each GPU still sees
# cfg.batch_size samples per step — the per-GPU workload doesn't change.
# Why scale the batch at all? To actually utilise both GPUs in parallel.
# With batch_size=128 and 2 GPUs, DataParallel would give each GPU only 64
# samples — half the arithmetic intensity, leaving Tensor Cores half-idle.
#
# Linear scaling rule (Goyal et al., 2017): when global batch size doubles,
# double the LR. The gradient per sample is unchanged — the optimizer step
# represents the same total curvature — so the LR must scale to keep the
# effective step size constant.
#
# Single GPU: batch=128, lr=3e-4 (baseline)
# T4x2:       batch=256, lr=6e-4 (each GPU still sees 128 samples/step)
total_batch = cfg.batch_size * N_GPUS
scaled_lr   = cfg.learning_rate * N_GPUS

print(f'Batch size  : {cfg.batch_size} per GPU × {N_GPUS} GPU(s) = {total_batch} total')
print(f'LR          : {cfg.learning_rate:.1e} base × {N_GPUS} = {scaled_lr:.1e} scaled')

# ── Data ──────────────────────────────────────────────────────────────────────
train_tf, val_tf = get_transforms()
train_ds = PCamDataset(TRAIN_X, TRAIN_Y, train_tf)
val_ds   = PCamDataset(VAL_X,   VAL_Y,   val_tf)

loader_kwargs = dict(
    batch_size=total_batch,      # total across all GPUs
    num_workers=cfg.num_workers,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)

# ── Model setup ───────────────────────────────────────────────────────────────
model = build_model().to(device)

if N_GPUS > 1:
    # DataParallel replicates the model on each GPU, runs forward/backward in
    # parallel, then reduces (sums) gradients back to GPU:0 for the update.
    # Simpler than DistributedDataParallel (DDP) and works in a notebook
    # without multiprocessing setup. DDP is more efficient for large models
    # (avoids the GPU:0 bottleneck) but overkill for ResNet-18 on 2 GPUs.
    model = nn.DataParallel(model)
    print(f'Using DataParallel across {N_GPUS} GPUs')
else:
    # torch.compile (PyTorch 2.0+): traces the computation graph and applies
    # kernel fusion and other optimisations via TorchInductor backend.
    # OLD WAY: eager execution — every op dispatched individually.
    # With compile: ops are fused (e.g. conv + BN + ReLU in a single kernel),
    # reducing kernel launch overhead and memory round-trips.
    # Typical gains: 10–20% on a single T4. First epoch is slower (compile
    # overhead ~30–60 s) — all subsequent epochs benefit.
    # Not combined with DataParallel to avoid interaction issues in 2.4.x.
    model = torch.compile(model)
    print('Using torch.compile (single GPU — first epoch includes compile overhead)')

# ── Optimiser and scheduler ───────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=scaled_lr,                  # scaled for multi-GPU
    weight_decay=cfg.weight_decay,
)

# Cosine annealing: decays LR smoothly from scaled_lr to eta_min over epochs.
# Used in DINO, MAE, SimCLR, and most modern vision pipelines. Eliminates
# manual step_size/gamma hyperparameters of StepLR.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6
)

# GradScaler for AMP — automatically manages loss scaling factor.
scaler = torch.amp.GradScaler('cuda')

# ── Training loop ─────────────────────────────────────────────────────────────
best_auc         = 0.0
patience_counter = 0
all_metrics      = []

print(f'\n  {"Ep":>3}  {"Train L":>8}  {"Val L":>8}  {"Acc":>7}  {"AUC":>7}  {"F1":>7}  {"LR":>9}')
print(f'  {"-"*3}  {"-"*8}  {"-"*8}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*9}')

for epoch in range(1, cfg.epochs + 1):
    log.info(f'Epoch {epoch}/{cfg.epochs}')
    train_loss  = train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, cfg)
    val_metrics = evaluate(model, val_loader, criterion, device)
    current_lr  = scheduler.get_last_lr()[0]

    record = {k: v for k, v in val_metrics.items() if not k.startswith('_')}
    record.update({'epoch': epoch, 'train_loss': round(train_loss, 4), 'lr': round(current_lr, 8)})
    all_metrics.append(record)

    print(f"  {epoch:>3}  {train_loss:>8.4f}  {val_metrics['loss']:>8.4f}  "
          f"{val_metrics['accuracy']:>7.4f}  {val_metrics['auc']:>7.4f}  "
          f"{val_metrics['f1']:>7.4f}  {current_lr:>9.2e}")

    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        patience_counter = 0
        # DataParallel wraps the model — save the inner module's state_dict
        # so the checkpoint is portable (loadable without DataParallel).
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state, OUTPUT_DIR / 'best_model.pt')
        np.save(OUTPUT_DIR / 'best_val_probs.npy',  val_metrics['_probs'])
        np.save(OUTPUT_DIR / 'best_val_labels.npy', val_metrics['_labels'])
        log.info(f'  → best AUC={best_auc:.4f} — checkpoint saved')
    else:
        patience_counter += 1
        log.info(f'  → no improvement ({patience_counter}/{cfg.patience})')
        if patience_counter >= cfg.patience:
            log.info(f'Early stopping triggered at epoch {epoch}')
            break

state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save(state, OUTPUT_DIR / 'final_model.pt')
with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

print(f'\nTraining complete. Best AUC: {best_auc:.4f}')

In [ ]:
# ── 8. Threshold optimisation ─────────────────────────────────────────────────
#
# OLD WAY: use threshold=0.5 (the default for argmax on softmax outputs).
# PROBLEM: 0.5 is arbitrary. For an imbalanced task or when false negatives
# are more costly than false positives (as in cancer screening), 0.5 is almost
# never the right operating point.
#
# In our previous run (0.5 threshold): 31% false negative rate — 1 in 3 tumours
# predicted as normal. Unacceptable for any medical application.
#
# THIS APPROACH — two clinically meaningful thresholds:
#
# 1. Youden's J: maximises sensitivity + specificity - 1.
#    Finds the point on the ROC curve furthest from the diagonal.
#    Standard method in diagnostic medicine (Youden, 1950).
#
# 2. 95% sensitivity threshold: the HIGHEST threshold that still keeps
#    sensitivity >= 95%. This maximises specificity (fewest false alarms)
#    while meeting the sensitivity floor required for clinical screening.
#    Searching low→high and taking the last match gives the correct answer.
#    (Taking the first match — a common mistake — returns threshold=0.01
#    with 100% sensitivity and 0% specificity, which is trivially useless.)

val_probs  = np.load(OUTPUT_DIR / 'best_val_probs.npy')
val_labels = np.load(OUTPUT_DIR / 'best_val_labels.npy')

thresholds    = np.linspace(0.01, 0.99, 500)
j_scores      = []
sensitivities = []
specificities = []

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    tp = ((preds == 1) & (val_labels == 1)).sum()
    tn = ((preds == 0) & (val_labels == 0)).sum()
    fp = ((preds == 1) & (val_labels == 0)).sum()
    fn = ((preds == 0) & (val_labels == 1)).sum()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivities.append(sens)
    specificities.append(spec)
    j_scores.append(sens + spec - 1)

best_j_idx    = int(np.argmax(j_scores))
best_t_youden = float(thresholds[best_j_idx])

# Walk low→high; keep overwriting — final value is the HIGHEST threshold
# where sensitivity is still >= 95%. This gives maximum specificity at the
# sensitivity floor, which is the clinically useful operating point.
sens_95_idx = None
for i, s in enumerate(sensitivities):
    if s >= 0.95:
        sens_95_idx = i

threshold_results = {
    'default_threshold_0.5': {
        'note': 'arbitrary — included for comparison only'
    },
    'youden_optimal': {
        'threshold':   round(best_t_youden, 4),
        'sensitivity': round(sensitivities[best_j_idx], 4),
        'specificity': round(specificities[best_j_idx], 4),
        'j_statistic': round(j_scores[best_j_idx], 4),
    },
    'sensitivity_95pct': {
        'threshold':   round(float(thresholds[sens_95_idx]), 4) if sens_95_idx is not None else None,
        'sensitivity': round(sensitivities[sens_95_idx], 4) if sens_95_idx is not None else None,
        'specificity': round(specificities[sens_95_idx], 4) if sens_95_idx is not None else None,
        'note': 'highest threshold still achieving >= 95% sensitivity (maximises specificity at sensitivity floor)'
    },
}

with open(OUTPUT_DIR / 'threshold.json', 'w') as f:
    json.dump(threshold_results, f, indent=2)

print('Threshold analysis:')
print(json.dumps(threshold_results, indent=2))

In [ ]:
# ── 9. Artifacts ──────────────────────────────────────────────────────────────
print('Artifacts in', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:35s}  {f.stat().st_size / 1e6:.1f} MB')

print('\nBest epoch metrics:')
best = max(all_metrics, key=lambda x: x['auc'])
print(json.dumps({k: v for k, v in best.items() if k != 'confusion_matrix'}, indent=2))